Creation of database

In [ ]:
import sqlite3
import os
from faker import Faker
import random

fake = Faker()

def generate_row():
    """Generate a row of data for the large database."""
    return (
        fake.uuid4(),
        fake.name(),
        fake.email(),
        fake.address(),
        fake.phone_number(),
        fake.job(),
        fake.company(),
        fake.date_of_birth().strftime('%Y-%m-%d'),
        fake.ssn(),
        fake.credit_card_number(),
        fake.credit_card_expire(),
        fake.credit_card_provider(),
        fake.country(),            # bank_country
        fake.currency_code(),
        round(random.uniform(100, 10000), 2),
        fake.date_time_this_decade().strftime('%Y-%m-%d %H:%M:%S')
    )

def create_large_database(db_name, target_size_bytes):
    conn = sqlite3.connect(db_name)
    cur = conn.cursor()

    # Create the full schema with primary key
    cur.execute('''
        CREATE TABLE IF NOT EXISTS customers (
            id TEXT PRIMARY KEY,
            name TEXT,
            email TEXT,
            address TEXT,
            phone_number TEXT,
            job TEXT,
            company TEXT,
            date_of_birth TEXT,
            ssn TEXT,
            credit_card_number TEXT,
            credit_card_expire TEXT,
            credit_card_provider TEXT,
            bank_country TEXT,
            currency_code TEXT,
            amount REAL,
            transaction_date TEXT
        )
    ''')
    conn.commit()

    count = 0
    while os.path.getsize(db_name) < target_size_bytes:
        row = generate_row()
        cur.execute('''
            INSERT INTO customers (
                id, name, email, address, phone_number, job, company, date_of_birth, ssn,
                credit_card_number, credit_card_expire, credit_card_provider, bank_country,
                currency_code, amount, transaction_date
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        ''', row)
        count += 1
        if count % 1000 == 0:
            conn.commit()
            print(f"{db_name}: Inserted {count} rows; size: {os.path.getsize(db_name)} bytes")

    conn.commit()
    print(f"{db_name} created with {count} rows. Final size: {os.path.getsize(db_name)} bytes")
    conn.close()



# Target sizes
target_sizes = {
    'large_database.db': 1073741824,     # 1 GB
    
}

# Create large database with full schema
print(f"\nCreating large_database.db...")
create_large_database('large_database.db', target_sizes['large_database.db'])


Creating sampled databases IDS


In [2]:
import sqlite3
import os
import random

def get_average_row_size(db_path, table_name='customers'):
    if not os.path.exists(db_path):
        raise FileNotFoundError(f"Database file {db_path} not found.")
    db_size = os.path.getsize(db_path)
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    cur.execute(f"SELECT COUNT(*) FROM {table_name}")
    row_count = cur.fetchone()[0]
    conn.close()
    if row_count == 0:
        raise ValueError("No rows in the table.")
    return db_size / row_count

def sample_ids_only(large_db_path, sample_db_map, avg_row_size_bytes):
    conn = sqlite3.connect(large_db_path)
    cur = conn.cursor()
    cur.execute("SELECT id FROM customers")
    all_ids = [row[0] for row in cur.fetchall()]
    conn.close()
    random.shuffle(all_ids) #shuffles all id

    for db_name, size_mb in sample_db_map.items():
        target_rows = int((size_mb * 1024 * 1024) / avg_row_size_bytes)
        print(f"Creating {db_name} with {target_rows} sampled IDs...")

        conn_sample = sqlite3.connect(db_name)
        cur_sample = conn_sample.cursor()
        cur_sample.execute("DROP TABLE IF EXISTS sample_ids")
        cur_sample.execute("CREATE TABLE sample_ids (id TEXT PRIMARY KEY)")
        for i in range(target_rows):
            cur_sample.execute("INSERT INTO sample_ids (id) VALUES (?)", (all_ids[i],))
            if i % 10000 == 0:
                conn_sample.commit()
        conn_sample.commit()
        conn_sample.close()


# Run both phases
sample_targets_mb = {
    '50mb_sample.db':50,
    '100mb_sample.db':100,
    '150mb_sample.db':150,
    '200mb_sample.db':200,
    '250mb_sample.db':250,
    '750mb_sample.db':750,
    '500mb_sample.db':500,
    '1000mb_sample.db':1000
    
}

avg_row_size = get_average_row_size('large_database.db')
sample_ids_only('large_database.db', sample_targets_mb, avg_row_size)

Creating 1000mb_sample.db with 1953805 sampled IDs...


Enrching sampled databases

In [ ]:
import sqlite3
import os
from tqdm import tqdm

def enrich_sampled_dbs_with_full_rows(large_db_path, sample_db_paths, id_table="sample_ids"):
    insert_query = '''
        INSERT INTO customers (
            id, name, email, address, phone_number, job, company, date_of_birth,
            ssn, credit_card_number, credit_card_expire, credit_card_provider,
            bank_country, currency_code, amount, transaction_date
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    '''
    batch_size = 500  # Avoid "too many variables" error

    for sample_db in sample_db_paths:
        print(f"\n🔄 Enriching {sample_db} with full rows...")

        # Step 1: Read IDs from sample DB
        conn_sample = sqlite3.connect(sample_db)
        cur_sample = conn_sample.cursor()
        cur_sample.execute(f"SELECT id FROM {id_table}")
        sampled_ids = [row[0] for row in cur_sample.fetchall()]

        # Step 2: Create 'customers' table if not exists
        cur_sample.execute('DROP TABLE IF EXISTS customers')
        cur_sample.execute('''
            CREATE TABLE customers (
                id TEXT PRIMARY KEY,
                name TEXT,
                email TEXT,
                address TEXT,
                phone_number TEXT,
                job TEXT,
                company TEXT,
                date_of_birth TEXT,
                ssn TEXT,
                credit_card_number TEXT,
                credit_card_expire TEXT,
                credit_card_provider TEXT,
                bank_country TEXT,
                currency_code TEXT,
                amount REAL,
                transaction_date TEXT
            )
        ''')
        conn_sample.commit()

        # Step 3: Fetch full rows from large DB in batches
        conn_large = sqlite3.connect(large_db_path)
        cur_large = conn_large.cursor()

        all_rows = []
        for i in tqdm(range(0, len(sampled_ids), batch_size), desc=f"Fetching from {sample_db}"):
            batch = sampled_ids[i:i + batch_size]
            placeholders = ','.join(['?'] * len(batch))
            cur_large.execute(f"SELECT * FROM customers WHERE id IN ({placeholders})", batch)
            all_rows.extend(cur_large.fetchall())
        conn_large.close()

        # Step 4: Insert into sample DB in one transaction
        cur_sample.execute("BEGIN TRANSACTION")
        cur_sample.executemany(insert_query, all_rows)
        cur_sample.execute("COMMIT")
        conn_sample.close()

        print(f"✅ Finished {sample_db} with {len(all_rows)} full rows inserted.")

# ✅ Sample Usage
sample_db_paths = [

    
    '500mb_sample.db',
    '750mb_sa
    '1000mb_sample.db'
    
]

enrich_sampled_dbs_with_full_rows('large_database.db', sample_db_paths)



🔄 Enriching 500mb_sample.db with full rows...


Fetching from 500mb_sample.db: 100%|██████████| 1954/1954 [12:19<00:00,  2.64it/s]


✅ Finished 500mb_sample.db with 976902 full rows inserted.

🔄 Enriching 1000mb_sample.db with full rows...


Fetching from 1000mb_sample.db:  97%|█████████▋| 3780/3908 [24:13<00:45,  2.81it/s]

LLM to SQL

In [6]:
import requests
import json
import sqlite3
import time
import csv
import os
from tqdm import tqdm

# API setup
url = "http://localhost:8000/v1/chat/completions"
headers = {"Content-Type": "application/json"}

execution_summary = []
nl_sql_log = []

def ask_model(prompt):
    data = {
        "model": "defog/llama-3-sqlcoder-8b",
        "temperature": 0.0001,
        "messages": [{"role": "user", "content": prompt}]
    }
    response = requests.post(url, headers=headers, data=json.dumps(data))
    if response.status_code == 200:
        model_output = response.json()["choices"][0]["message"]["content"]
        print("\n🔍 Model Raw Output:\n", model_output)
        return model_output.strip()
    else:
        raise Exception(f"API request failed: {response.status_code}")

def natural_language_to_sql(user_command):
    prompt = f"""
### Task:
Convert the following natural language command into a correctly formatted SQLite SQL query.

### Notes:
- Use SQLite-compatible functions only.
- Avoid EXTRACT, AGE, TO_DATE, DATE_PART, etc.
- Use julianday for date math if needed.
- Return ONLY the SQL query (no markdown, no explanation).

Table: customers
Columns:
  id, name, email, address, phone_number, job, company, date_of_birth,
  ssn, credit_card_number, credit_card_expire, credit_card_provider,
  bank_country, currency_code, amount, transaction_date

### User Input:
{user_command}

### SQL Output:
"""
    sql = ask_model(prompt)
    nl_sql_log.append([user_command, sql])
    if sql.strip().upper().startswith("SELECT") and sql.strip().endswith(";"):
        return sql.strip()
    else:
        print("⚠️ Warning: Invalid SQL returned. Using fallback.")
        return "SELECT * FROM customers LIMIT 5;"

def execute_sql(db_path, sql_query):
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    start = time.time()
    cur.execute(sql_query)
    result = cur.fetchall()
    end = time.time()
    conn.close()
    return result, end - start

def aggregate_results(results):
    try:
        return sum(row[0] for row in results if isinstance(row[0], (int, float)))
    except:
        return None

def compute_relative_error(gold, sample):
    try:
        return abs(sample - gold) / gold * 100 if gold != 0 else None
    except:
        return None

def compute_speed_overhead(gold_time, sample_time):
    
        return (sample_time) 
    

def export_summary_results():
    with open("experiment_results.csv", "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            "Query", "Database", "Raw Sample Result", "Scaled Result",
            "Execution Time (s)", "Gold Execution Time (s)",
            "Relative Error (%)", "Speed Overhead (%)"
        ])
        writer.writerows(execution_summary)

def export_query_txt():
    with open("queries_log.txt", "w") as f:
        for nl, sql in nl_sql_log:
            f.write(f"Natural Language: {nl}\nSQL Query: {sql}\n\n")

def export_error_overhead():
    with open("errors_overhead.csv", "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["Query", "Database", "Relative Error (%)", "Speed Overhead (%)"])
        for row in execution_summary:
            writer.writerow([row[0], row[1], row[6], row[7]])

def run_experiment(sql_query, large_db_path, sample_db_paths_with_size):
    print("\n🔍 Executing on full database...")
    gold_results, gold_time = execute_sql(large_db_path, sql_query)
    gold_aggregated = aggregate_results(gold_results)

    print(f"✅ Full DB Result: {gold_aggregated} | Execution Time: {gold_time:.4f} seconds")

    for db_path, size_bytes in tqdm(sample_db_paths_with_size.items(), desc="🔁 Sample DBs"):
        scaling_factor = size_bytes / os.path.getsize(large_db_path)
        inverse_scaling = 1 / scaling_factor

        try:
            sample_results, sample_time = execute_sql(db_path, sql_query)
            raw_result = aggregate_results(sample_results)

            scaled_result = raw_result * inverse_scaling if raw_result is not None else None
            rel_error = compute_relative_error(gold_aggregated, scaled_result)
            speed_overhead = compute_speed_overhead(gold_time, sample_time)

            print(f"\n📁 {db_path}")
            print(f"Raw: {raw_result}, Scaled: {scaled_result:.2f} | Sample Time: {sample_time:.4f}s")
            print(f"Relative Error: {rel_error:.2f}%" if rel_error is not None else "N/A")
            print(f"Speed Overhead: {speed_overhead:.2f}%" if speed_overhead is not None else "N/A")

            execution_summary.append([
                sql_query,
                db_path,
                raw_result,
                scaled_result,
                f"{sample_time:.4f}",
                f"{gold_time:.4f}",
                f"{rel_error:.2f}" if rel_error is not None else None,
                f"{speed_overhead:.2f}" if speed_overhead is not None else None
            ])

        except Exception as e:
            print(f"❌ Failed on {db_path}: {e}")
            execution_summary.append([sql_query, db_path, None, None, None, f"{gold_time:.4f}", None, None])

def main():
    large_db_path = "large_database.db"
    sample_db_paths_with_size = {
        "50mb_sample.db": 52428800,
        "100mb_sample.db": 104857600,
        "150mb_sample.db": 157286400,
        "200mb_sample.db": 209715200,
        "250mb_sample.db": 262144000,
        "500mb_sample.db": 524288000,
        "750mb_sample.db": 786432000,
        "1000mb_sample.db": 1048576000
    }

    while True:
        user_input = input("\n🧠 Enter your query (or type 'exit'): ")
        if user_input.lower() == "exit":
            break

        sql_query = natural_language_to_sql(user_input)
        print(f"\n📝 SQL Query:\n{sql_query}")
        run_experiment(sql_query, large_db_path, sample_db_paths_with_size)

    export_summary_results()
    export_query_txt()
    export_error_overhead()

    print("\n✅ Saved:")
    print(" - experiment_results.csv (full details)")
    print(" - queries_log.txt (natural input + SQL)")
    print(" - errors_overhead.csv (relative error + speed overhead only)")

if __name__ == "__main__":
    main()



🧠 Enter your query (or type 'exit'):  How many customers are in the database?



🔍 Model Raw Output:
 SELECT COUNT(*) FROM customers;

📝 SQL Query:
SELECT COUNT(*) FROM customers;

🔍 Executing on full database...
✅ Full DB Result: 15404000 | Execution Time: 1.2329 seconds


🔁 Sample DBs:  12%|█▎        | 1/8 [00:00<00:01,  5.03it/s]


📁 50mb_sample.db
Raw: 97690, Scaled: 15403957.63 | Sample Time: 0.1968s
Relative Error: 0.00%
Speed Overhead: 0.20%


🔁 Sample DBs:  25%|██▌       | 2/8 [00:00<00:01,  3.31it/s]

...


📁 750mb_sample.db
Raw: 0, Scaled: 0.00 | Sample Time: 4.5329s
N/A
Speed Overhead: 4.53%


🔁 Sample DBs: 100%|██████████| 8/8 [00:17<00:00,  2.24s/it]


📁 1000mb_sample.db
Raw: 0, Scaled: 0.00 | Sample Time: 5.8510s
N/A
Speed Overhead: 5.85%



🧠 Enter your query (or type 'exit'):  exit



✅ Saved:
 - experiment_results.csv (full details)
 - queries_log.txt (natural input + SQL)
 - errors_overhead.csv (relative error + speed overhead only)


In [15]:
import sqlite3
import os
import random
from tqdm import tqdm

SOURCE_DB = "large_database.db"

TARGETS = {
    "500mb_sample.db": 500 * 1024 * 1024,
    "750mb_sample.db": 750 * 1024 * 1024,
    "1000mb_sample.db": 1000 * 1024 * 1024,
}

SAMPLE_FOR_ESTIMATION = 1000  # rows to sample for row size estimate

def create_table(conn):
    cur = conn.cursor()
    cur.execute("""
        CREATE TABLE IF NOT EXISTS customers (
            id INTEGER PRIMARY KEY,
            name TEXT,
            email TEXT,
            address TEXT,
            phone_number TEXT,
            job TEXT,
            company TEXT,
            date_of_birth TEXT,
            ssn TEXT,
            credit_card_number TEXT,
            credit_card_expire TEXT,
            credit_card_provider TEXT,
            bank_country TEXT,
            currency_code TEXT,
            amount REAL,
            transaction_date TEXT
        )
    """)
    conn.commit()

def get_all_ids(source_conn):
    cur = source_conn.cursor()
    cur.execute("SELECT id FROM customers")
    return [row[0] for row in cur.fetchall()]

def estimate_average_row_size(source_conn):
    cur = source_conn.cursor()
    cur.execute("SELECT * FROM customers LIMIT ?", (SAMPLE_FOR_ESTIMATION,))
    rows = cur.fetchall()

    temp_db = "temp_estimate.db"
    if os.path.exists(temp_db):
        os.remove(temp_db)

    with sqlite3.connect(temp_db) as conn:
        create_table(conn)
        insert_ids(conn, [row[0] for row in rows])  # Insert IDs first
        enrich_rows(conn, rows)  # Then enrich with full data
        conn.commit()

    size = os.path.getsize(temp_db)
    os.remove(temp_db)
    return size / len(rows)

def insert_ids(target_conn, ids):
    cur = target_conn.cursor()
    cur.executemany("INSERT INTO customers (id) VALUES (?)", [(i,) for i in ids])
    target_conn.commit()

def enrich_rows(target_conn, rows):
    cur = target_conn.cursor()
    for row in rows:
        cur.execute("""
            UPDATE customers SET
                name = ?, email = ?, address = ?, phone_number = ?, job = ?, company = ?,
                date_of_birth = ?, ssn = ?, credit_card_number = ?, credit_card_expire = ?,
                credit_card_provider = ?, bank_country = ?, currency_code = ?, amount = ?,
                transaction_date = ?
            WHERE id = ?
        """, (
            row[1], row[2], row[3], row[4], row[5], row[6], row[7], row[8], row[9],
            row[10], row[11], row[12], row[13], row[14], row[15], row[0]
        ))
    target_conn.commit()

def fetch_rows_by_ids(source_conn, id_list):
    cur = source_conn.cursor()
    batch_size = 1000
    all_rows = []

    for i in range(0, len(id_list), batch_size):
        batch_ids = id_list[i:i+batch_size]
        placeholders = ','.join('?' for _ in batch_ids)
        query = f"SELECT * FROM customers WHERE id IN ({placeholders})"
        cur.execute(query, batch_ids)
        all_rows.extend(cur.fetchall())

    return all_rows

def generate_sample_db(db_name, target_size_bytes, source_conn, avg_row_size):
    print(f"\n📁 Creating {db_name} to reach ~{target_size_bytes // (1024 * 1024)}MB")

    estimated_rows = int(target_size_bytes / avg_row_size)
    print(f"📊 Estimated rows needed: {estimated_rows}")

    all_ids = get_all_ids(source_conn)
    sampled_ids = random.sample(all_ids, estimated_rows)
    rows = fetch_rows_by_ids(source_conn, sampled_ids)

    target_conn = sqlite3.connect(db_name)
    create_table(target_conn)

    insert_ids(target_conn, sampled_ids)

    with tqdm(total=len(rows), desc=f"🚀 Enriching {db_name}", unit="rows") as pbar:
        for i in range(0, len(rows), 1000):
            enrich_rows(target_conn, rows[i:i+1000])
            pbar.update(len(rows[i:i+1000]))

    target_conn.close()
    final_size = os.path.getsize(db_name)
    print(f"✅ Done: {db_name} is {(final_size / (1024 * 1024)):.2f} MB")

if __name__ == "__main__":
    source_conn = sqlite3.connect(SOURCE_DB)
    avg_row_size = estimate_average_row_size(source_conn)

    for db_name, size in TARGETS.items():
        generate_sample_db(db_name, size, source_conn, avg_row_size)

    source_conn.close()


IntegrityError: datatype mismatch